# WSI watershed connected-component audit and post-resolution cleanup

Run (or safely reuse) one globally normalized, watershed-resolved InstanSeg WSI pass over the validated SLIDE-0330 all-channel half crop, then audit and apply the provisional post-resolution cleanup policy to its model-resolution Zarr. The cleanup uses exact 8-connectivity, applies the strict model-scale component rule `component_pixels > 10` independently to each nuclear component, rejects coordinated IDs with no surviving nuclear component, retains only cell components touching a surviving nucleus, and rejects all unnucleated cells. Diagonal-only objects remain because diagonal contact is connected under 8-connectivity. A separate cleaned Zarr, per-ID metrics, JSON summary, and max-pooled DAPI removal overview are written under `connectedness_audit`; the resolved input is never overwritten. The original audit helper below remains available for comparison, while the final cell invokes the rerunnable cleanup.

In [ ]:
from pathlib import Path
import json, re, subprocess, sys, time
import numpy as np
import pandas as pd
import tifffile
import zarr
from skimage.measure import label as label_equal_values
from tqdm.auto import tqdm

INSTANSEG_ROOT = Path('/data1/lowes/ratnayn/Codex/projects/instanseg')
CROP = Path('/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/SLIDE-0330_all_channels_half_crop.ome.tif')
OUTPUT_DIR = CROP.parent / 'connectedness_audit'
RESOLVED_ZARR = OUTPUT_DIR / 'SLIDE-0330_watershed_resolved.zarr'
PER_CELL_CSV = OUTPUT_DIR / 'SLIDE-0330_connectedness_per_cell.csv'
SUMMARY_CSV = OUTPUT_DIR / 'SLIDE-0330_connectedness_summary.csv'
SUMMARY_JSON = OUTPUT_DIR / 'SLIDE-0330_connectedness_summary.json'
# Safe rerun default: reuse the completed on-disk Zarr without launching inference.
# Set this to True only if the Zarr is absent and a fresh GPU pass is intended.
RUN_WSI = False
RUN_CONNECTEDNESS_AUDIT = True
REUSE_COMPLETED_WSI = True
WRITE_PER_CELL_CSV = True
AUDIT_CHUNK = 2048
MODEL_NAME = 'fluorescence_nuclei_and_cells'
PIXEL_SIZE_UM = 0.325
NORMALIZATION_PERCENTILES = (0.1, 99.9)
WSI_TILE_SIZE = 2048
WSI_OVERLAP = 80
WSI_DETECTION_SIZE = 20
WSI_BATCH_SIZE = 1
SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_F480_D2S9R_555', 'R9_CD68_E3O7V_488',
    'R12_CD3E_E4T1B_AF555',
]
REFERENCE_CHANNEL = 'R1_DAPI'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(INSTANSEG_ROOT) not in sys.path:
    sys.path.insert(0, str(INSTANSEG_ROOT))
print({'crop': str(CROP), 'resolved_zarr': str(RESOLVED_ZARR), 'run_wsi': RUN_WSI, 'run_audit': RUN_CONNECTEDNESS_AUDIT})

In [ ]:
if not CROP.is_file():
    raise FileNotFoundError(CROP)
import instanseg
from instanseg import InstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as inference_class
inference_class.TiffSlide = TiffSlide
fork_path = Path(instanseg.__file__).resolve()
fork_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=INSTANSEG_ROOT, text=True).strip()
fork_status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=INSTANSEG_ROOT, text=True).strip()
assert fork_path.is_relative_to(INSTANSEG_ROOT), fork_path
assert hasattr(InstanSeg, 'eval_whole_slide_image_global_normalization')
with tifffile.TiffFile(CROP) as tif:
    series = tif.series[0]
    crop_shape, crop_axes = tuple(int(v) for v in series.shape), series.axes
    ome_xml = tif.ome_metadata or ''
channel_names = [m.group(1) for m in re.finditer(r'<(?:[^:>]+:)?Channel\b[^>]*?Name=\"([^\"]*)\"', ome_xml)]
if crop_axes != 'CYX' or len(channel_names) != crop_shape[0]:
    raise ValueError(f'Expected named CYX input, got axes={crop_axes}, shape={crop_shape}, names={len(channel_names)}')
if len(channel_names) != len(set(channel_names)):
    raise ValueError('OME channel names are not unique.')
lookup = {name: index for index, name in enumerate(channel_names)}
missing = [name for name in SEGMENTATION_CHANNELS if name not in lookup]
if missing:
    raise KeyError(f'Missing segmentation channels: {missing}')
CHANNEL_IDS = [lookup[name] for name in SEGMENTATION_CHANNELS]
REFERENCE_CHANNEL_ID = lookup[REFERENCE_CHANNEL]
assert CHANNEL_IDS[0] == REFERENCE_CHANNEL_ID
print({'python': sys.executable, 'instanseg': str(fork_path), 'commit': fork_commit, 'dirty': bool(fork_status), 'crop_shape': crop_shape, 'channel_ids': CHANNEL_IDS})

In [ ]:
def compatible_completed_output(path):
    if not path.is_dir():
        return False
    try:
        arr = zarr.open(str(path), mode='r')
        attrs = dict(arr.attrs)
        settings = attrs.get('wsi_settings') or {}
        normalization = attrs.get('normalization') or {}
        resolution = attrs.get('resolution') or {}
        return (
            attrs.get('status') == 'complete'
            and list(attrs.get('planes', [])) == ['nuclei', 'cells']
            and Path(attrs.get('source_image', '')).resolve() == CROP.resolve()
            and list(attrs.get('channel_ids', [])) == CHANNEL_IDS
            and settings.get('tile_size') == WSI_TILE_SIZE
            and settings.get('overlap') == WSI_OVERLAP
            and settings.get('detection_size') == WSI_DETECTION_SIZE
            and settings.get('resolve_cell_and_nucleus') is True
            and settings.get('resolution_method') == 'watershed'
            and [float(v) for v in normalization.get('percentiles', [])] == list(NORMALIZATION_PERCENTILES)
            and resolution.get('method') == 'watershed'
            and resolution.get('allow_unnucleated_cells') is True
        )
    except Exception:
        return False

if compatible_completed_output(RESOLVED_ZARR) and REUSE_COMPLETED_WSI:
    print('Reusing compatible completed WSI:', RESOLVED_ZARR)
elif RUN_WSI:
    if RESOLVED_ZARR.exists():
        raise ValueError(f'Existing output is incompatible; move it or choose a new path: {RESOLVED_ZARR}')
    model = InstanSeg(MODEL_NAME, verbosity=1)
    started = time.perf_counter()
    observed = model.eval_whole_slide_image_global_normalization(
        str(CROP), channel_ids=CHANNEL_IDS, pixel_size=PIXEL_SIZE_UM,
        normalization_percentiles=NORMALIZATION_PERCENTILES,
        reference_channel_id=REFERENCE_CHANNEL_ID, tile_size=WSI_TILE_SIZE,
        overlap=WSI_OVERLAP, detection_size=WSI_DETECTION_SIZE,
        batch_size=WSI_BATCH_SIZE, output_path=RESOLVED_ZARR, overwrite=False,
        resolve_cell_and_nucleus=True, resolution_method='watershed',
        allow_unnucleated_cells=True, cleanup_fragments=True, seed_threshold=0.6,
    )
    assert Path(observed).resolve() == RESOLVED_ZARR.resolve()
    print(f'WSI inference and watershed completed in {(time.perf_counter()-started)/60:.1f} min')
else:
    raise RuntimeError('No compatible result exists; set RUN_WSI=True.')
assert compatible_completed_output(RESOLVED_ZARR)

In [ ]:
def connected_component_table(resolved, chunk_size=2048):
    cells, nuclei = resolved[1], resolved[0]
    height, width = map(int, cells.shape)
    parent, area, nuclear_pixels, label_id = [], [], [], []

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(first, second):
        root_a, root_b = find(int(first)), find(int(second))
        if root_a == root_b:
            return
        if label_id[root_a] != label_id[root_b]:
            raise RuntimeError('Attempted to union different cell IDs.')
        if area[root_a] < area[root_b]:
            root_a, root_b = root_b, root_a
        parent[root_b] = root_a
        area[root_a] += area[root_b]
        nuclear_pixels[root_a] += nuclear_pixels[root_b]

    def union_boundary(old_labels, old_nodes, new_labels, new_nodes):
        mask = (old_labels > 0) & (old_labels == new_labels)
        if not np.any(mask):
            return
        pairs = np.unique(np.stack((old_nodes[mask], new_nodes[mask]), axis=1), axis=0)
        for first, second in pairs:
            union(first, second)

    x_starts = list(range(0, width, chunk_size))
    y_starts = list(range(0, height, chunk_size))
    upper_labels, upper_nodes = [None] * len(x_starts), [None] * len(x_starts)
    for y0 in tqdm(y_starts, desc='Connectedness rows'):
        y1 = min(y0 + chunk_size, height)
        left_labels = left_nodes = None
        for column, x0 in enumerate(x_starts):
            x1 = min(x0 + chunk_size, width)
            cell_chunk = np.asarray(cells[y0:y1, x0:x1], dtype=np.int32)
            nucleus_chunk = np.asarray(nuclei[y0:y1, x0:x1], dtype=np.int32)
            components = label_equal_values(cell_chunk, background=0, connectivity=1)
            count = int(components.max())
            if count:
                component_ids, first_indices, counts = np.unique(components, return_index=True, return_counts=True)
                keep = component_ids > 0
                component_ids, first_indices, counts = component_ids[keep], first_indices[keep], counts[keep]
                component_labels = cell_chunk.ravel()[first_indices].astype(np.int64)
                if np.any(component_labels <= 0):
                    raise RuntimeError('Foreground component received a background label.')
                overlap = ((cell_chunk == nucleus_chunk) & (cell_chunk > 0)).ravel()
                component_nuclear = np.bincount(components.ravel(), weights=overlap, minlength=count + 1)[1:].astype(np.int64)
                offset = len(parent)
                for local_index in range(count):
                    node = offset + local_index
                    parent.append(node)
                    area.append(int(counts[local_index]))
                    nuclear_pixels.append(int(component_nuclear[local_index]))
                    label_id.append(int(component_labels[local_index]))
                node_map = np.where(components > 0, components.astype(np.int64) - 1 + offset, -1)
            else:
                node_map = np.full(components.shape, -1, dtype=np.int64)
            if left_labels is not None:
                union_boundary(left_labels, left_nodes, cell_chunk[:, 0], node_map[:, 0])
            if upper_labels[column] is not None:
                union_boundary(upper_labels[column], upper_nodes[column], cell_chunk[0, :], node_map[0, :])
            left_labels, left_nodes = cell_chunk[:, -1].copy(), node_map[:, -1].copy()
            upper_labels[column], upper_nodes[column] = cell_chunk[-1, :].copy(), node_map[-1, :].copy()

    roots = [index for index in range(len(parent)) if find(index) == index]
    return pd.DataFrame({
        'cell_id': [label_id[index] for index in roots],
        'component_pixels': [area[index] for index in roots],
        'nuclear_pixels': [nuclear_pixels[index] for index in roots],
    })

In [ ]:
if not RUN_CONNECTEDNESS_AUDIT:
    raise RuntimeError('Set RUN_CONNECTEDNESS_AUDIT=True to calculate metrics.')
resolved = zarr.open(str(RESOLVED_ZARR), mode='r')
attrs = dict(resolved.attrs)
validation = attrs.get('validation') or {}
assert attrs.get('status') == 'complete' and validation.get('nuclear_cell_ids_agree') is True
started = time.perf_counter()
components = connected_component_table(resolved, AUDIT_CHUNK)
components['touches_assigned_nucleus'] = components['nuclear_pixels'] > 0
components['pixels_without_nucleus_contact'] = np.where(components['touches_assigned_nucleus'], 0, components['component_pixels'])
per_cell = components.groupby('cell_id', sort=True).agg(
    component_count=('cell_id', 'size'),
    total_pixels=('component_pixels', 'sum'),
    largest_component_pixels=('component_pixels', 'max'),
    nucleus_touching_components=('touches_assigned_nucleus', 'sum'),
    pixels_outside_nucleus_components=('pixels_without_nucleus_contact', 'sum'),
).reset_index()
per_cell['secondary_component_pixels'] = per_cell['total_pixels'] - per_cell['largest_component_pixels']
per_cell['secondary_component_fraction'] = per_cell['secondary_component_pixels'] / per_cell['total_pixels']
per_cell['is_disconnected'] = per_cell['component_count'] > 1
nuclear_max, cell_max = [int(v) for v in attrs['max_label_by_plane']]
proxy_count = int(validation.get('proxy_cells', 0))
proxy_start = nuclear_max - proxy_count + 1
per_cell['category'] = np.select(
    [per_cell['cell_id'] > nuclear_max, per_cell['cell_id'] >= proxy_start],
    ['unnucleated', 'proxy'], default='associated_nucleated',
)
per_cell.loc[per_cell['category'] == 'unnucleated', 'pixels_outside_nucleus_components'] = np.nan
category_summary = per_cell.groupby('category', sort=False).agg(
    cells=('cell_id', 'size'),
    disconnected_cells=('is_disconnected', 'sum'),
    total_pixels=('total_pixels', 'sum'),
    secondary_component_pixels=('secondary_component_pixels', 'sum'),
    pixels_outside_nucleus_components=('pixels_outside_nucleus_components', 'sum'),
).reset_index()
category_summary['disconnected_fraction'] = category_summary['disconnected_cells'] / category_summary['cells']
category_summary['secondary_pixel_fraction'] = category_summary['secondary_component_pixels'] / category_summary['total_pixels']
overall = {
    'cells': int(len(per_cell)),
    'components': int(len(components)),
    'disconnected_cells': int(per_cell['is_disconnected'].sum()),
    'disconnected_fraction': float(per_cell['is_disconnected'].mean()),
    'secondary_component_pixels': int(per_cell['secondary_component_pixels'].sum()),
    'secondary_pixel_fraction': float(per_cell['secondary_component_pixels'].sum() / per_cell['total_pixels'].sum()),
    'maximum_components_for_one_cell': int(per_cell['component_count'].max()),
    'resolver_ambiguous_parent_count': int(validation.get('ambiguous_cells', 0)),
    'resolver_proxy_count': proxy_count,
    'resolver_unseeded_parent_pixels_preserved': int(validation.get('unseeded_parent_pixels_preserved', 0)),
    'audit_connectivity': 4,
    'elapsed_minutes': float((time.perf_counter() - started) / 60),
}
if WRITE_PER_CELL_CSV:
    per_cell.to_csv(PER_CELL_CSV, index=False)
category_summary.to_csv(SUMMARY_CSV, index=False)
SUMMARY_JSON.write_text(json.dumps({'overall': overall, 'categories': category_summary.to_dict(orient='records')}, indent=2) + '\n')
display(pd.DataFrame([overall]))
display(category_summary)
display(per_cell.loc[per_cell['is_disconnected']].sort_values(['secondary_component_pixels', 'component_count'], ascending=False).head(25))
print({'per_cell_csv': str(PER_CELL_CSV) if WRITE_PER_CELL_CSV else None, 'summary_csv': str(SUMMARY_CSV), 'summary_json': str(SUMMARY_JSON)})

In [ ]:
# Plot the largest disconnected associated-nucleated and proxy examples.
# This rescans the model-resolution cell Zarr for bounding boxes but does not rerun inference.
import matplotlib.pyplot as plt
from skimage.segmentation import find_boundaries

EXAMPLES_PER_CATEGORY = 6
DISPLAY_MIN_NATIVE_PIXELS = 384
DISPLAY_MAX_NATIVE_PIXELS = 768
DISPLAY_PADDING_NATIVE_PIXELS = 96
example_rows = pd.concat([
    per_cell.loc[per_cell['is_disconnected'] & per_cell['category'].eq(category)]
    .nlargest(EXAMPLES_PER_CATEGORY, 'secondary_component_pixels')
    for category in ('associated_nucleated', 'proxy')
], ignore_index=True)
if example_rows.empty:
    raise RuntimeError('No disconnected associated-nucleated or proxy cells were found.')
example_ids = set(example_rows['cell_id'].astype(int))
model_height, model_width = map(int, resolved.shape[-2:])
native_height, native_width = map(int, crop_shape[-2:])
bounds = {cell_id: [model_height, model_width, -1, -1] for cell_id in example_ids}
for y0 in tqdm(range(0, model_height, AUDIT_CHUNK), desc='Finding example bounds'):
    y1 = min(y0 + AUDIT_CHUNK, model_height)
    for x0 in range(0, model_width, AUDIT_CHUNK):
        x1 = min(x0 + AUDIT_CHUNK, model_width)
        block = np.asarray(resolved[1, y0:y1, x0:x1])
        present = example_ids.intersection(int(value) for value in np.unique(block) if value > 0)
        for cell_id in present:
            yy, xx = np.nonzero(block == cell_id)
            item = bounds[cell_id]
            item[0] = min(item[0], y0 + int(yy.min()))
            item[1] = min(item[1], x0 + int(xx.min()))
            item[2] = max(item[2], y0 + int(yy.max()))
            item[3] = max(item[3], x0 + int(xx.max()))

def centered_window(lower, upper, limit):
    lower = int(np.floor(lower)) - DISPLAY_PADDING_NATIVE_PIXELS
    upper = int(np.ceil(upper)) + DISPLAY_PADDING_NATIVE_PIXELS
    requested = min(DISPLAY_MAX_NATIVE_PIXELS, max(DISPLAY_MIN_NATIVE_PIXELS, upper - lower))
    center = (lower + upper) // 2
    start = max(0, min(center - requested // 2, limit - requested))
    return start, min(limit, start + requested)

def model_indices(native_start, native_stop, model_size, native_size):
    coordinates = np.arange(native_start, native_stop, dtype=np.int64)
    indices = ((2 * coordinates + 1) * model_size) // (2 * native_size)
    return np.clip(indices, 0, model_size - 1)

with tifffile.TiffFile(str(CROP)) as handle:
    input_store = handle.series[0].aszarr(level=0)
    try:
        input_array = zarr.open(input_store, mode='r')
        fig, axes = plt.subplots(len(example_rows), 3, figsize=(15, 4.5 * len(example_rows)), squeeze=False)
        for row_axes, row in zip(axes, example_rows.itertuples(index=False)):
            cell_id = int(row.cell_id)
            my0, mx0, my1, mx1 = bounds[cell_id]
            ny0, ny1 = centered_window(my0 * native_height / model_height, (my1 + 1) * native_height / model_height, native_height)
            nx0, nx1 = centered_window(mx0 * native_width / model_width, (mx1 + 1) * native_width / model_width, native_width)
            native_dapi = np.asarray(input_array.oindex[REFERENCE_CHANNEL_ID, slice(ny0, ny1), slice(nx0, nx1)], dtype=np.float32)
            low, high = np.percentile(native_dapi, (1.0, 99.8))
            dapi_display = np.clip((native_dapi - low) / max(high - low, 1e-6), 0, 1)
            model_y, model_x = model_indices(ny0, ny1, model_height, native_height), model_indices(nx0, nx1, model_width, native_width)
            sy0, sx0 = int(model_y.min()), int(model_x.min())
            label_block = np.asarray(resolved[:, sy0:int(model_y.max()) + 1, sx0:int(model_x.max()) + 1])
            label_view = np.take(np.take(label_block, model_y - sy0, axis=1), model_x - sx0, axis=2)
            for axis in row_axes:
                axis.imshow(dapi_display, cmap='gray', interpolation='nearest')
                axis.axis('off')
            # Combined view: target cell and assigned nucleus together.
            row_axes[0].contour(find_boundaries(label_view[1], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.25, alpha=0.3)
            row_axes[0].contour(find_boundaries(label_view[1] == cell_id, mode='outer'), levels=[0.5], colors=['magenta'], linewidths=1.5)
            if np.any(label_view[0] == cell_id):
                row_axes[0].contour(find_boundaries(label_view[0] == cell_id, mode='outer'), levels=[0.5], colors=['cyan'], linewidths=1.2)
            # Nuclear-only view: all nuclei faintly cyan and the assigned nucleus in red.
            row_axes[1].contour(find_boundaries(label_view[0], mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.25, alpha=0.35)
            if np.any(label_view[0] == cell_id):
                row_axes[1].contour(find_boundaries(label_view[0] == cell_id, mode='outer'), levels=[0.5], colors=['red'], linewidths=1.5)
            # Cell-only view: all cells faintly yellow and the target cell in magenta.
            row_axes[2].contour(find_boundaries(label_view[1], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.25, alpha=0.35)
            row_axes[2].contour(find_boundaries(label_view[1] == cell_id, mode='outer'), levels=[0.5], colors=['magenta'], linewidths=1.5)
            details = f'{row.category} | ID {cell_id} | 4-components={row.component_count}, secondary={row.secondary_component_pixels}px ({row.secondary_component_fraction:.1%})'
            row_axes[0].set_title('Combined\n' + details, fontsize=8)
            row_axes[1].set_title('Nuclear mask only', fontsize=9)
            row_axes[2].set_title('Cell mask only', fontsize=9)
    finally:
        input_store.close()
fig.suptitle('Disconnected-cell examples | combined: magenta cell/cyan nucleus | nucleus-only: red target | cell-only: magenta target', fontsize=12)
fig.tight_layout()
plt.show()
display(example_rows[['cell_id', 'category', 'component_count', 'total_pixels', 'secondary_component_pixels', 'secondary_component_fraction', 'nucleus_touching_components', 'pixels_outside_nucleus_components']])

In [ ]:
# Representative gallery of the distinct topology categories discussed above.
# IDs were selected from the completed crop using exact 8-connectivity.
CATEGORY_EXAMPLES = {
    'Diagonal-only: 4-split but 8-connected': [17891, 185988, 513250, 609913, 240201, 619733, 331916, 648752, 632665, 616574],
    'True fragmented proxy nucleus/cell': [583987, 592003, 593743, 593288, 592290, 592294, 576076, 592730, 580746, 577018],
    'Associated nucleus with 8-disconnected remnant': [23689, 235083, 327566, 493230, 311188, 478800, 502461, 56012, 128676, 177542],
    'Associated cell with nucleus-free island': [322305, 67479, 509566, 279818, 520135, 178956, 205935, 9336, 132176, 110746],
    'Associated cell: every component has nuclear pixels': [493230, 327566, 527554, 318811, 23689, 502461, 550948, 529428, 482404, 385153],
    'Fragmented unnucleated cell': [643521, 617804, 607218, 599507, 646773, 607878, 630377, 622950, 604921, 636366],
}
gallery_ids = {cell_id for ids in CATEGORY_EXAMPLES.values() for cell_id in ids}
gallery_bounds = {cell_id: [model_height, model_width, -1, -1] for cell_id in gallery_ids}
for y0 in tqdm(range(0, model_height, AUDIT_CHUNK), desc='Finding gallery bounds'):
    y1 = min(y0 + AUDIT_CHUNK, model_height)
    for x0 in range(0, model_width, AUDIT_CHUNK):
        x1 = min(x0 + AUDIT_CHUNK, model_width)
        block = np.asarray(resolved[:, y0:y1, x0:x1])
        present = gallery_ids.intersection(int(value) for value in np.unique(block) if value > 0)
        for cell_id in present:
            yy, xx = np.nonzero(np.any(block == cell_id, axis=0))
            item = gallery_bounds[cell_id]
            item[0] = min(item[0], y0 + int(yy.min()))
            item[1] = min(item[1], x0 + int(xx.min()))
            item[2] = max(item[2], y0 + int(yy.max()))
            item[3] = max(item[3], x0 + int(xx.max()))

# Reapply the model-scale >min_size cutoff to final 8-connected components.
INSTANSEG_MIN_SIZE = 10

def filter_small_8_components(mask, min_size):
    components = label_equal_values(mask.astype(np.uint8), background=0, connectivity=2)
    if components.max() == 0:
        return mask.copy(), 0, []
    counts = np.bincount(components.ravel())
    keep_ids = np.flatnonzero(counts > int(min_size))
    keep_ids = keep_ids[keep_ids > 0]
    keep_set = set(int(value) for value in keep_ids)
    removed_areas = [int(counts[index]) for index in range(1, len(counts)) if index not in keep_set]
    return np.isin(components, keep_ids), int(components.max()), removed_areas

def proposed_topology_resolution(nucleus_mask, cell_mask, category):
    nucleus_mask, cell_mask = nucleus_mask.astype(bool), cell_mask.astype(bool)
    nucleus_area = int(nucleus_mask.sum())
    retained_nucleus, nucleus_components, removed_nuclear_areas = filter_small_8_components(nucleus_mask, INSTANSEG_MIN_SIZE)
    if category == 'Fragmented unnucleated cell':
        return np.zeros_like(nucleus_mask), np.zeros_like(cell_mask), 'reject unnucleated cell'
    if nucleus_area > 0 and not np.any(retained_nucleus):
        return np.zeros_like(nucleus_mask), np.zeros_like(cell_mask), f'reject nucleus+cell: no component > {INSTANSEG_MIN_SIZE}px'
    action = 'unchanged'
    if removed_nuclear_areas:
        action = f'remove nuclear components <= {INSTANSEG_MIN_SIZE}px: {removed_nuclear_areas}'
    cell_components = label_equal_values(cell_mask.astype(np.uint8), background=0, connectivity=2)
    retained_cell = cell_mask.copy()
    if cell_components.max() > 1 and np.any(retained_nucleus):
        keep_ids = np.unique(cell_components[retained_nucleus & cell_mask])
        keep_ids = keep_ids[keep_ids > 0]
        retained_cell = np.isin(cell_components, keep_ids)
        if np.any(retained_cell != cell_mask):
            action = action + '; remove nucleus-free cell components' if action != 'unchanged' else 'remove nucleus-free cell components'
    return retained_nucleus, retained_cell, action

with tifffile.TiffFile(str(CROP)) as handle:
    input_store = handle.series[0].aszarr(level=0)
    try:
        input_array = zarr.open(input_store, mode='r')
        for category, ids in CATEGORY_EXAMPLES.items():
            fig, axes = plt.subplots(len(ids), 4, figsize=(20, 3.6 * len(ids)), squeeze=False)
            for row_axes, cell_id in zip(axes, ids):
                my0, mx0, my1, mx1 = gallery_bounds[cell_id]
                if my1 < 0:
                    raise RuntimeError(f'Gallery ID {cell_id} was not found.')
                ny0, ny1 = centered_window(my0 * native_height / model_height, (my1 + 1) * native_height / model_height, native_height)
                nx0, nx1 = centered_window(mx0 * native_width / model_width, (mx1 + 1) * native_width / model_width, native_width)
                native_dapi = np.asarray(input_array.oindex[REFERENCE_CHANNEL_ID, slice(ny0, ny1), slice(nx0, nx1)], dtype=np.float32)
                low, high = np.percentile(native_dapi, (1.0, 99.8))
                dapi_display = np.clip((native_dapi - low) / max(high - low, 1e-6), 0, 1)
                model_y = model_indices(ny0, ny1, model_height, native_height)
                model_x = model_indices(nx0, nx1, model_width, native_width)
                sy0, sx0 = int(model_y.min()), int(model_x.min())
                label_block = np.asarray(resolved[:, sy0:int(model_y.max()) + 1, sx0:int(model_x.max()) + 1])
                label_view = np.take(np.take(label_block, model_y - sy0, axis=1), model_x - sx0, axis=2)
                target_nucleus = label_view[0] == cell_id
                target_cell = label_view[1] == cell_id
                proposed_nucleus_model, proposed_cell_model, proposed_action = proposed_topology_resolution(label_block[0] == cell_id, label_block[1] == cell_id, category)
                proposed_nucleus = np.take(np.take(proposed_nucleus_model, model_y - sy0, axis=0), model_x - sx0, axis=1)
                proposed_cell = np.take(np.take(proposed_cell_model, model_y - sy0, axis=0), model_x - sx0, axis=1)
                for axis in row_axes:
                    axis.imshow(dapi_display, cmap='gray', interpolation='nearest')
                    axis.axis('off')
                # Combined context and target outlines.
                row_axes[0].contour(find_boundaries(label_view[0], mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.2, alpha=0.25)
                row_axes[0].contour(find_boundaries(label_view[1], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.2, alpha=0.25)
                if np.any(target_nucleus):
                    row_axes[0].contour(find_boundaries(target_nucleus, mode='outer'), levels=[0.5], colors=['cyan'], linewidths=1.4)
                if np.any(target_cell):
                    row_axes[0].contour(find_boundaries(target_cell, mode='outer'), levels=[0.5], colors=['magenta'], linewidths=1.4)
                # Nuclear mask in isolation.
                row_axes[1].contour(find_boundaries(label_view[0], mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.2, alpha=0.25)
                if np.any(target_nucleus):
                    row_axes[1].imshow(np.ma.masked_where(~target_nucleus, target_nucleus), cmap='Reds', alpha=0.45, interpolation='nearest')
                    row_axes[1].contour(find_boundaries(target_nucleus, mode='outer'), levels=[0.5], colors=['red'], linewidths=1.4)
                # Cell mask in isolation.
                row_axes[2].contour(find_boundaries(label_view[1], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.2, alpha=0.25)
                if np.any(target_cell):
                    row_axes[2].imshow(np.ma.masked_where(~target_cell, target_cell), cmap='RdPu', alpha=0.35, interpolation='nearest')
                    row_axes[2].contour(find_boundaries(target_cell, mode='outer'), levels=[0.5], colors=['magenta'], linewidths=1.4)
                # Proposed resolution: remove the original target from context, then draw only retained pixels.
                context_nuclei = label_view[0].copy()
                context_cells = label_view[1].copy()
                context_nuclei[target_nucleus] = 0
                context_cells[target_cell] = 0
                row_axes[3].contour(find_boundaries(context_nuclei, mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.2, alpha=0.2)
                row_axes[3].contour(find_boundaries(context_cells, mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.2, alpha=0.2)
                if np.any(proposed_nucleus):
                    row_axes[3].contour(find_boundaries(proposed_nucleus, mode='outer'), levels=[0.5], colors=['cyan'], linewidths=1.4)
                if np.any(proposed_cell):
                    row_axes[3].contour(find_boundaries(proposed_cell, mode='outer'), levels=[0.5], colors=['magenta'], linewidths=1.4)
                if not np.any(proposed_nucleus) and not np.any(proposed_cell):
                    # Keep the rejected cell visible for QC, but deliberately show no nuclear mask.
                    if np.any(target_cell):
                        row_axes[3].imshow(np.ma.masked_where(~target_cell, target_cell), cmap='Reds', alpha=0.18, interpolation='nearest')
                        row_axes[3].contour(find_boundaries(target_cell, mode='outer'), levels=[0.5], colors=['red'], linewidths=1.4, linestyles='--')
                    row_axes[3].text(0.5, 0.04, 'REJECTED CELL (shown without nucleus)', color='red', fontsize=9, fontweight='bold', ha='center', va='bottom', transform=row_axes[3].transAxes)
                row_axes[0].set_title(f'Combined | ID {cell_id}', fontsize=9)
                row_axes[1].set_title('Nuclear mask only' if np.any(target_nucleus) else 'Nuclear mask absent', fontsize=9)
                row_axes[2].set_title('Cell mask only', fontsize=9)
                row_axes[3].set_title('Proposed resolution\n' + proposed_action, fontsize=8)
            fig.suptitle(category, fontsize=13)
            fig.tight_layout()
            plt.show()
    finally:
        input_store.close()
print('Gallery categories:', list(CATEGORY_EXAMPLES))

In [ ]:
# Apply the provisional post-resolution policy to the existing resolved Zarr.
# This cell never reruns inference and never overwrites RESOLVED_ZARR.
import sys
_notebook_dir = Path.cwd() if (Path.cwd() / 'instanseg_connectedness_cleanup.py').is_file() else Path.cwd() / 'notebooks'
if _notebook_dir.is_dir() and str(_notebook_dir) not in sys.path:
    sys.path.insert(0, str(_notebook_dir))
from instanseg_connectedness_cleanup import run_cleanup, INSTANSEG_MIN_SIZE
CLEANED_ZARR = OUTPUT_DIR / 'SLIDE-0330_watershed_resolved_postresolution_8conn_min10.zarr'
CLEANUP_METRICS_CSV = OUTPUT_DIR / 'SLIDE-0330_postresolution_cleanup_per_id.csv'
CLEANUP_METRICS_JSON = OUTPUT_DIR / 'SLIDE-0330_postresolution_cleanup_summary.json'
CLEANUP_OVERVIEW_PNG = OUTPUT_DIR / 'SLIDE-0330_postresolution_cleanup_removed_overview.png'
CLEANUP_REUSE = True
CLEANUP_OVERWRITE = False
if not RUN_CONNECTEDNESS_AUDIT:
    raise RuntimeError('Set RUN_CONNECTEDNESS_AUDIT=True to apply cleanup and calculate metrics.')
cleanup_summary = run_cleanup(
    RESOLVED_ZARR, CLEANED_ZARR, CLEANUP_METRICS_CSV, CLEANUP_METRICS_JSON,
    CLEANUP_OVERVIEW_PNG, CROP, reference_channel_id=REFERENCE_CHANNEL_ID,
    native_shape=crop_shape[-2:], chunk_size=AUDIT_CHUNK,
    min_size=INSTANSEG_MIN_SIZE, reuse=CLEANUP_REUSE, overwrite=CLEANUP_OVERWRITE,
)
display(pd.DataFrame([cleanup_summary['original'], cleanup_summary['final']]))
display(pd.DataFrame([{k: cleanup_summary[k] for k in (
    'removed_nuclear_components', 'removed_nuclear_pixels',
    'rejected_coordinated_ids', 'rejected_coordinated_nuclear_pixels',
    'rejected_coordinated_cell_pixels', 'removed_nucleus_free_cell_components',
    'removed_nucleus_free_cell_pixels', 'rejected_unnucleated_ids',
    'rejected_unnucleated_cell_pixels', 'removed_cell_pixels_total')}]))
print({k: cleanup_summary[k] for k in ('cleaned_zarr', 'metrics_csv', 'metrics_json', 'overview_png')})

In [ ]:
# Visual QC: stratified examples of complete unnucleated cells rejected by cleanup.
# Cyan = every predicted nucleus nearby; magenta/red = the target cell.
# Override with a list of IDs to inspect particular cells.
from scipy.ndimage import distance_transform_edt
from skimage.segmentation import find_boundaries
UNNUCLEATED_EXAMPLE_IDS = None
UNNUCLEATED_EXAMPLES = 12
UNNUCLEATED_GALLERY_PNG = OUTPUT_DIR / 'SLIDE-0330_rejected_unnucleated_cell_gallery.png'
cleanup_per_id = pd.read_csv(CLEANUP_METRICS_CSV)
unnucleated = cleanup_per_id.loc[cleanup_per_id['category'].eq('rejected_unnucleated')].copy()
assert len(unnucleated) == cleanup_summary['rejected_unnucleated_ids']
assert (unnucleated['original_nuclear_pixels'] == 0).all()
if UNNUCLEATED_EXAMPLE_IDS is None:
    ordered = unnucleated.sort_values('original_cell_pixels').reset_index(drop=True)
    selected_rows = ordered.iloc[np.linspace(0, len(ordered) - 1, UNNUCLEATED_EXAMPLES).round().astype(int)]
else:
    selected_rows = unnucleated.set_index('label_id').loc[[int(v) for v in UNNUCLEATED_EXAMPLE_IDS]].reset_index()
selected_ids = set(selected_rows['label_id'].astype(int))
model_height, model_width = map(int, resolved.shape[-2:])
native_height, native_width = map(int, crop_shape[-2:])
selected_bounds = {label_id: [model_height, model_width, -1, -1] for label_id in selected_ids}
for y0 in tqdm(range(0, model_height, AUDIT_CHUNK), desc='Finding unnucleated-cell bounds'):
    y1 = min(y0 + AUDIT_CHUNK, model_height)
    for x0 in range(0, model_width, AUDIT_CHUNK):
        x1 = min(x0 + AUDIT_CHUNK, model_width)
        block = np.asarray(resolved[1, y0:y1, x0:x1])
        for label_id in selected_ids.intersection(int(v) for v in np.unique(block) if v > 0):
            yy, xx = np.nonzero(block == label_id)
            bound = selected_bounds[label_id]
            bound[0] = min(bound[0], y0 + int(yy.min())); bound[1] = min(bound[1], x0 + int(xx.min()))
            bound[2] = max(bound[2], y0 + int(yy.max())); bound[3] = max(bound[3], x0 + int(xx.max()))
assert all(bound[2] >= 0 for bound in selected_bounds.values())
def unnucleated_window(lower, upper, limit, padding=128, minimum=512, maximum=768):
    lower, upper = int(np.floor(lower)) - padding, int(np.ceil(upper)) + padding
    width = min(maximum, max(minimum, upper - lower))
    center = (lower + upper) // 2
    start = max(0, min(center - width // 2, limit - width))
    return start, min(limit, start + width)
def unnucleated_model_indices(start, stop, model_size, native_size):
    native = np.arange(start, stop, dtype=np.int64)
    return np.clip(((2 * native + 1) * model_size) // (2 * native_size), 0, model_size - 1)
with tifffile.TiffFile(str(CROP)) as handle:
    input_store = handle.series[0].aszarr(level=0)
    try:
        input_array = zarr.open(input_store, mode='r')
        fig, axes = plt.subplots(len(selected_rows), 4, figsize=(16, 4 * len(selected_rows)), squeeze=False)
        for row_axes, row in zip(axes, selected_rows.itertuples(index=False)):
            label_id, cell_area = int(row.label_id), int(row.original_cell_pixels)
            my0, mx0, my1, mx1 = selected_bounds[label_id]
            ny0, ny1 = unnucleated_window(my0 * native_height / model_height, (my1 + 1) * native_height / model_height, native_height)
            nx0, nx1 = unnucleated_window(mx0 * native_width / model_width, (mx1 + 1) * native_width / model_width, native_width)
            dapi = np.asarray(input_array.oindex[REFERENCE_CHANNEL_ID, slice(ny0, ny1), slice(nx0, nx1)], dtype=np.float32)
            low, high = np.percentile(dapi, (1.0, 99.8))
            dapi = np.clip((dapi - low) / max(high - low, 1e-6), 0, 1)
            model_y = unnucleated_model_indices(ny0, ny1, model_height, native_height)
            model_x = unnucleated_model_indices(nx0, nx1, model_width, native_width)
            sy0, sx0 = int(model_y.min()), int(model_x.min())
            labels = np.asarray(resolved[:, sy0:int(model_y.max()) + 1, sx0:int(model_x.max()) + 1])
            model_target = labels[1] == label_id
            model_nuclei = labels[0] > 0
            overlap = int(np.count_nonzero(model_target & model_nuclei))
            nearest = float(distance_transform_edt(~model_nuclei)[model_target].min()) if model_nuclei.any() else float('inf')
            view = np.take(np.take(labels, model_y - sy0, axis=1), model_x - sx0, axis=2)
            target = view[1] == label_id
            nuclei = view[0] > 0
            for axis in row_axes:
                axis.imshow(dapi, cmap='gray', interpolation='nearest'); axis.axis('off')
            row_axes[0].contour(find_boundaries(nuclei, mode='outer'), [0.5], colors=['cyan'], linewidths=0.5)
            row_axes[0].contour(find_boundaries(target, mode='outer'), [0.5], colors=['magenta'], linewidths=1.5)
            row_axes[0].set_title(f'Combined | ID {label_id} | area={cell_area}px')
            row_axes[1].set_title('DAPI only')
            row_axes[2].contour(find_boundaries(target, mode='outer'), [0.5], colors=['magenta'], linewidths=1.5)
            row_axes[2].set_title('Original cell mask')
            row_axes[3].contour(find_boundaries(nuclei, mode='outer'), [0.5], colors=['cyan'], linewidths=0.8)
            row_axes[3].contour(find_boundaries(target, mode='outer'), [0.5], colors=['red'], linewidths=1.2, linestyles='--')
            row_axes[3].set_title(f'All nearby nuclei | inside cell={overlap}px | nearest={nearest:.1f} model px')
        fig.suptitle('Stratified rejected unnucleated cells: cyan nuclei, target cell magenta/red', fontsize=14)
        fig.tight_layout(); fig.savefig(UNNUCLEATED_GALLERY_PNG, dpi=160, bbox_inches='tight'); plt.show()
    finally:
        input_store.close()
print({'gallery': str(UNNUCLEATED_GALLERY_PNG), 'ids': sorted(selected_ids)})

In [ ]:
# Zoom into the fixed-grid region containing the most removed cell pixels.
from instanseg_connectedness_cleanup import make_removed_cell_hotspot_preview
REMOVAL_HOTSPOT_PNG = OUTPUT_DIR / 'SLIDE-0330_removed_cell_hotspot_preview.png'
cleaned = zarr.open(str(CLEANED_ZARR), mode='r')
hotspot_summary = make_removed_cell_hotspot_preview(
    CROP, resolved, cleaned, REMOVAL_HOTSPOT_PNG,
    reference_channel_id=REFERENCE_CHANNEL_ID, native_shape=crop_shape[-2:],
    metrics_csv=CLEANUP_METRICS_CSV, search_block=192, read_chunk=AUDIT_CHUNK,
)
print(hotspot_summary)